In [ ]:
! pip install pymilvus
! pip install sentence_transformers

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ZILLIZ_TOKEN = user_secrets.get_secret("zilliz_api_key")
ZILLIZ_ENDPOINT = user_secrets.get_secret("Zilliz_endpoint")


In [ ]:
import pandas as pd
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

In [ ]:
# 1. Load the data
# Note: In Kaggle, the path includes the dataset name
FILE_PATH = '/kaggle/input/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_2k/HDFS_2k.log_structured.csv'
df = pd.read_csv(FILE_PATH)
print(df.shape)
print(df.head(5))

In [ ]:
def smart_truncate(text, max_len=2980):
    if len(text) <= max_len:
        return text
    # Keep the first 1000 chars and the last 1000 chars
    return f"{text[:1000]}...[TRUNCATED]...{text[-1000:]}"


In [ ]:
# client = MilvusClient(
#     uri=ZILLIZ_ENDPOINT,
#     token=ZILLIZ_TOKEN
# )

# client.delete(
#     collection_name="logs_collection",
#     filter="id >= 0"
# )

# print("Data cleared from logs_collection (Schema preserved).")

In [ ]:
# 2. Setup Zilliz Client
client = MilvusClient(
    uri=ZILLIZ_ENDPOINT,
    token=ZILLIZ_TOKEN
)

partition_name = "hdfs"

# Create the partition if it doesn't exist
if not client.has_partition(collection_name="logs_collection", partition_name=partition_name):
    client.create_partition(
        collection_name="logs_collection", 
        partition_name=partition_name
    )
    print(f"Partition '{partition_name}' created.")

# 3. Load Embedding Model (BERT-based)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 4. Prepare Data for Zilliz
data = []
print(f"Processing {len(df)} logs...")

# We process all 2,000 logs 
for idx, row in df.iterrows():
    
    vector = model.encode(row['Content']).tolist()
    
    # Combine Date and Time for the timestamp field
    full_timestamp = f"{row['Date']} {row['Time']}"

    truncated_content = smart_truncate(str(row['Content']))
    
    data.append({
        "vector": vector,
        "timestamp": str(full_timestamp),
        "log_level": str(row['Level']),
        "message": truncated_content
    })

# 5. Insert into Zilliz
client.insert(collection_name="logs_collection", data=data, partition_name=partition_name)

print("✅ Ingestion Complete! Data is now in Zilliz.")

In [ ]:
stats = client.get_collection_stats(collection_name="logs_collection")
print(f"Total Collection Count (Approx): {stats['row_count']}")

# 2. Try to 'Query' the partition specifically
# This forces the database to look inside the 'toronto_logs' folder
res = client.query(
    collection_name="logs_collection",
    filter="id >= 0",
    partition_names=["hdfs"], # Look ONLY here
    limit=5,
    output_fields=["message"]
)

if res:
    print(f"✅ Success! Found {len(res)} logs in 'hdfs' partition.")
    print(f"Sample: {res[0]['message']}")
else:
    print("❌ Partition is actually empty. Ingestion might have failed.")